# 03 · Backtest Analysis

Run the event-driven backtest and dig into the trade log, equity curve, and risk-agent decisions.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
import plotly.express as px
from src.data_pipeline import DataPipeline
from src.model import MicrostructureModel, build_signal_frame
from src.backtester import EventDrivenBacktester
from src.risk_agent import RiskMonitoringAgent
from src.utils import load_config, CostParams
cfg = load_config(os.path.abspath("../data/config.json"))
cfg

In [ ]:
pipe = DataPipeline(config=cfg); pipe.build()
bars = pipe.load_bars("uniswap_bars"); pipe.close()
model = MicrostructureModel(cfg["ma_window_bars"], cfg["vol_window_bars"],
                            cfg["z_threshold"], cfg["use_garch"])
model.fit(bars["midprice"].values[:int(len(bars)*0.5)])
sig = build_signal_frame(bars, model)

agent = RiskMonitoringAgent(cfg["max_portfolio_delta"], cfg["max_daily_drawdown_pct"],
                            cfg["stop_loss_pct"], cfg["vol_spike_multiple"], cfg["max_hold_bars"])
bt = EventDrivenBacktester(cfg["initial_capital_usd"], cfg["max_position_size_usd"],
                           CostParams.from_config(cfg), agent, cfg["bar_frequency"])
metrics = bt.run(sig)
metrics

## Equity curve

In [ ]:
res = bt.get_results()
eq = pd.DataFrame({"date": pd.to_datetime(res["equity_dates"]), "equity": res["equity_curve"]})
px.line(eq, x="date", y="equity", title="Equity curve (USD)")

## Trade-level P&L distribution

In [ ]:
tl = pd.DataFrame(res["trade_log"])
px.histogram(tl, x="pnl", nbins=60, title="Per-trade net P&L ($) — note left tail from whipsaws")

## Exit-reason breakdown

In [ ]:
tl['exit_reason'].value_counts()

## Risk-agent decisions

In [ ]:
dl = pd.DataFrame(res["decision_log"])
dl.groupby(["decision_type", "decision"]).size()

In [ ]:
dl[dl["decision"] == "REJECT"].head(10)